###  Text Preprocessing Pipeline
A complete preprocessing system that takes raw text input and outputs clean, structured text ready for machine learning models.
This pipeline should handle:
1. Cleaning raw text
2. Custom tokenization (not just default library use)
3. Stopword removal
4. Lemmatization
5. Output formatted processed text


In [19]:
#Step 1: import the required libraries for this project
# note: we will be using Spacy instead of NLTK beacuse of its efficiency, speed and ease of use for NLP tasks

import spacy
import re
import pandas as pd
import numpy as np
import warnings
import logging
import unicodedata
import html
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

# Load spaCy model for lemmatization
nlp = spacy.load("en_core_web_sm")

# English stopwords list
stop_words = set(stopwords.words('english'))

#remove warnings
import warnings
warnings.filterwarnings('ignore')

In [20]:
'''Logging alllows us to:
1.Track pipeline execution step-by-step
2.Debug errors in real systems
3.Monitor data flow in production
4.Store history of what happened internally'''

logging.basicConfig(
    filename="nlp_pipeline.log",   # log file
    filemode='a',           # append mode
    level=logging.INFO,        # log level
    format="%(asctime)s | %(levelname)s | %(message)s"
)

logger = logging.getLogger(__name__)

In [21]:
# Edge Case Handlers - these functions will be called before the main pipeline to handle specific edge cases
# Each function is designed to handle a specific type of edge case that may arise in real-world text data

# Edge Case 1: Handle None, empty string, or whitespace-only input
def handle_null_input(text):
    """Edge Case 1: Handle None, empty string, or whitespace-only input"""
    if text is None:
        return ""
    if not isinstance(text, str):
        return str(text) if text is not None else ""
    return text.strip()

# Edge Case 2: Handle Unicode, emojis, special characters
def handle_non_english_chars(text):
    """Edge Case 2: Handle Unicode, emojis, special characters"""
    # Normalize Unicode (e.g., é → e)
    text = unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')
    
    # Remove emojis
    emoji_pattern = re.compile(
        "["
        u"\U0001F600-\U0001F64F"  # emoticons
        u"\U0001F300-\U0001F5FF"  # symbols & pictographs
        u"\U0001F680-\U0001F6FF"  # transport & map symbols
        u"\U0001F1E0-\U0001F1FF"  # flags
        "]+", 
        flags=re.UNICODE
    )
    text = emoji_pattern.sub(r'', text)
    return text

# Edge Case 3: Handle HTML entities
def handle_html_entities(text):
    """Edge Case 3: Decode HTML entities (&amp; → &)"""
    return html.unescape(text)

# Edge Case 4: Handle URLs, emails, and social media handles
def handle_urls_emails(text):
    """Edge Case 4: Remove URLs, emails, and social media handles"""
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'@\w+|#\w+', '', text)
    return text

# Edge Case 5: Handle repeated characters (e.g., "coooool" → "cool")
def handle_repeated_characters(text):
    """Edge Case 5: Normalize repeated characters (coooool → cool)"""
    # Reduce 3+ repetitions to 2
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    return text

# Edge Case 6: Handle numbers in words (e.g., "ai2024" → "ai")
def handle_numbers_in_words(text):
    """Edge Case 6: Remove numbers from within words (ai2024 → ai)"""
    words = text.split()
    cleaned_words = []
    for word in words:
        if re.search(r'[a-zA-Z]', word) and re.search(r'\d', word):
            word = re.sub(r'\d+', '', word)
        cleaned_words.append(word)
    return ' '.join(cleaned_words)

# Edge Case 7: Handle special quotes and dashes
def handle_special_quotes_dashes(text):
    """Edge Case 7: Normalize smart quotes and special dashes"""
    text = text.replace('’', "'").replace('‘', "'").replace('”', '"').replace('“', '"')
    text = text.replace('—', '-').replace('–', '-')
    text = text.replace('…', ' ')
    return text

# Master Edge Case Handler - applies all edge case preprocessing BEFORE sending to main pipeline
def preprocess_edge_cases(text):
    """
    Master edge case handler - applies all edge case preprocessing
    BEFORE sending to main pipeline
    """
    if text is None or (isinstance(text, str) and text.strip() == ""):
        return ""
    
    if not isinstance(text, str):
        text = str(text)
    
    # Apply edge case handlers in logical order
    text = handle_null_input(text)
    text = handle_html_entities(text)
    text = handle_special_quotes_dashes(text)
    text = handle_urls_emails(text)
    text = handle_non_english_chars(text)
    text = handle_repeated_characters(text)
    text = handle_numbers_in_words(text)
    text = text.lower()  # ensure lowercase
    text = re.sub(r'[^a-z\s]', ' ', text)  # remove remaining special chars
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [22]:
# contractions dictionary, we will use this to expand contractions in the text, like "don't" to "do not", "can't" to "cannot"
contractions_dict = {
    "I'm": "I am",
    "I'm'a": "I am about to",
    "I'm'o": "I am going to",
    "I've": "I have",
    "I'll": "I will",
    "I'll've": "I will have",
    "I'd": "I would",
    "I'd've": "I would have",
    "Whatcha": "What are you",
    "amn't": "am not",
    "ain't": "are not",
    "aren't": "are not",
    "'cause": "because",
    "can't": "cannot",
    "can't've": "cannot have",
    "could've": "could have",
    "couldn't": "could not",
    "couldn't've": "could not have",
    "daren't": "dare not",
    "daresn't": "dare not",
    "dasn't": "dare not",
    "didn't": "did not",
    "didn’t": "did not",
    "don't": "do not",
    "don’t": "do not",
    "doesn't": "does not",
    "e'er": "ever",
    "everyone's": "everyone is",
    "finna": "fixing to",
    "gimme": "give me",
    "gon't": "go not",
    "gonna": "going to",
    "gotta": "got to",
    "hadn't": "had not",
    "hadn't've": "had not have",
    "hasn't": "has not",
    "haven't": "have not",
    "he've": "he have",
    "he's": "he is",
    "he'll": "he will",
    "he'll've": "he will have",
    "he'd": "he would",
    "he'd've": "he would have",
    "here's": "here is",
    "how're": "how are",
    "how'd": "how did",
    "how'd'y": "how do you",
    "how's": "how is",
    "how'll": "how will",
    "isn't": "is not",
    "it's": "it is",
    "'tis": "it is",
    "'twas": "it was",
    "it'll": "it will",
    "it'll've": "it will have",
    "it'd": "it would",
    "it'd've": "it would have",
    "kinda": "kind of",
    "let's": "let us",
    "luv": "love",
    "ma'am": "madam",
    "may've": "may have",
    "mayn't": "may not",
    "might've": "might have",
    "mightn't": "might not",
    "mightn't've": "might not have",
    "must've": "must have",
    "mustn't": "must not",
    "mustn't've": "must not have",
    "needn't": "need not",
    "needn't've": "need not have",
    "ne'er": "never",
    "o'": "of",
    "o'clock": "of the clock",
    "ol'": "old",
    "oughtn't": "ought not",
    "oughtn't've": "ought not have",
    "o'er": "over",
    "shan't": "shall not",
    "sha'n't": "shall not",
    "shalln't": "shall not",
    "shan't've": "shall not have",
    "she's": "she is",
    "she'll": "she will",
    "she'd": "she would",
    "she'd've": "she would have",
    "should've": "should have",
    "shouldn't": "should not",
    "shouldn't've": "should not have",
    "so've": "so have",
    "so's": "so is",
    "somebody's": "somebody is",
    "someone's": "someone is",
    "something's": "something is",
    "sux": "sucks",
    "that're": "that are",
    "that's": "that is",
    "that'll": "that will",
    "that'd": "that would",
    "that'd've": "that would have",
    "'em": "them",
    "there're": "there are",
    "there's": "there is",
    "there'll": "there will",
    "there'd": "there would",
    "there'd've": "there would have",
    "these're": "these are",
    "they're": "they are",
    "they've": "they have",
    "they'll": "they will",
    "they'll've": "they will have",
    "they'd": "they would",
    "they'd've": "they would have",
    "this's": "this is",
    "this'll": "this will",
    "this'd": "this would",
    "those're": "those are",
    "to've": "to have",
    "wanna": "want to",
    "wasn't": "was not",
    "we're": "we are",
    "we've": "we have",
    "we'll": "we will",
    "we'll've": "we will have",
    "we'd": "we would",
    "we'd've": "we would have",
    "weren't": "were not",
    "what're": "what are",
    "what'd": "what did",
    "what've": "what have",
    "what's": "what is",
    "what'll": "what will",
    "what'll've": "what will have",
    "when've": "when have",
    "when's": "when is",
    "where're": "where are",
    "where'd": "where did",
    "where've": "where have",
    "where's": "where is",
    "which's": "which is",
    "who're": "who are",
    "who've": "who have",
    "who's": "who is",
    "who'll": "who will",
    "who'll've": "who will have",
    "who'd": "who would",
    "who'd've": "who would have",
    "why're": "why are",
    "why'd": "why did",
    "why've": "why have",
    "why's": "why is",
    "will've": "will have",
    "won't": "will not",
    "won't've": "will not have",
    "would've": "would have",
    "wouldn't": "would not",
    "wouldn't've": "would not have",
    "y'all": "you all",
    "y'all're": "you all are",
    "y'all've": "you all have",
    "y'all'd": "you all would",
    "y'all'd've": "you all would have",
    "you're": "you are",
    "you've": "you have",
    "you'll've": "you shall have",
    "you'll": "you will",
    "you'd": "you would",
    "you'd've": "you would have",

    "to cause": "to cause",
    "will cause": "will cause",
    "should cause": "should cause",
    "would cause": "would cause",
    "can cause": "can cause",
    "could cause": "could cause",
    "must cause": "must cause",
    "might cause": "might cause",
    "shall cause": "shall cause",
    "may cause": "may cause"
}

In [23]:
# Function to validate input before processing, this is important to ensure that our pipeline does not break 
# when it encounters unexpected input types or formats

def validate_input(text):
    """
    STEP 0: Ensures safe pipeline execution
    MUST BE USED BEFORE ANY PROCESSING
    """

    if text is None:
        return False
    if not isinstance(text, str):
        return False
    if text.strip() == "":
        return False

    return True

In [24]:
# Function to expand contractions in the text, this is important to ensure that our pipeline can understand and process the text correctly, 
# as contractions can cause issues with tokenization and other NLP tasks

def expand_contractions(text):
    """
    Expands contractions like don't → do not
    """

    if not isinstance(text, str):
        return ""

    pattern = re.compile(r'\b(' + '|'.join(contractions_dict.keys()) + r')\b')

    return pattern.sub(lambda x: contractions_dict[x.group()], text)

In [25]:
# clean the text data by removing special characters, numbers, punctuation, and lowercasing the text
def clean_text(text):
    """
    Cleans raw text:
    - lowercase
    - remove URLs
    - remove emails
    - remove special characters
    - normalize spaces
    """

    if not isinstance(text, str) or text.strip() == "":
        return ""

    text = text.lower()

    # remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # remove emails
    text = re.sub(r'\S+@\S+', '', text)

    # keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)

    # remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [26]:
# creating a coustom tokenizer using regex to split the text into tokens (words)

def custom_tokenizer(text):
    """
    ADVANCED CUSTOM TOKENIZER (IMPROVED)
    - Handles alphanumeric tokens
    - Splits intelligently
    - Removes noise
    - Keeps meaningful single letters (a, i)
    """

    raw_tokens = re.findall(r'\b[a-zA-Z0-9]+\b', text)

    tokens = []

    for token in raw_tokens:
        token = token.lower()

        # remove numbers inside words (ai2024 → ai)
        token = re.sub(r'\d+', '', token)

        # skip empty after cleaning
        if token == "":
            continue

        # remove repeated chars (coooool → cool)
        token = re.sub(r'(.)\1{2,}', r'\1\1', token)

        # Keep single letters 'a' and 'i', remove other single letters
        if len(token) == 1 and token in ['a', 'i']:
            tokens.append(token)
        elif len(token) >= 2:
            if token not in ["thing", "something", "anything", "everything"]:
                tokens.append(token)

    return tokens

In [27]:
# we will remove punctuation here, its best here because it can creates problems later on real data.

def remove_stopwords(tokens):
    """
    Removes stopwords + noise filtering
    """

    filtered = []

    for t in tokens:
        t = t.lower().strip()

        if t in stop_words:
            continue

        if len(t) < 2:
            continue

        filtered.append(t)

    return filtered

In [28]:
# we will use spacy for lemmatization, its more efficient and accurate than nltk's lemmatizer, and it also handles edge cases better

def lemmatize_text(tokens):

    text = " ".join(tokens)

    doc = nlp(text, disable=["parser", "ner"])

    return [
        token.lemma_
        for token in doc
        if token.lemma_ != "-PRON-"
    ]

In [29]:
# Main pipeline function that applies all the steps in order, with logging and error handling

def text_preprocessing_pipeline(text, config=None):
    """
    ENHANCED TEXT PREPROCESSING PIPELINE
    Handles all edge cases automatically
    """
    if config is None:
        config = {
            "remove_stopwords": True,
            "lemmatize": True,
            "remove_duplicates": True
        }
    
    try:
        logger.info("Pipeline started")
        
        # Step 0: Validation
        if not validate_input(text):
            logger.warning("Invalid input received")
            return []
        
        logger.info(f"Raw Input: {text}")
        
        # ===== NEW: Edge case preprocessing =====
        text = preprocess_edge_cases(text)
        
        if text.strip() == "":
            logger.warning("Text became empty after edge case handling")
            return []
        # ========================================
        
        # Step 1: Expand contractions
        pattern = re.compile(r'\b(' + '|'.join(map(re.escape, contractions_dict.keys())) + r')\b')
        text = pattern.sub(lambda x: contractions_dict[x.group()], text)
        logger.info(f"After contraction expansion: {text}")
        
        # Step 2: Cleaning (already partially done in edge cases)
        text = text.lower()
        text = re.sub(r'http\S+|www\S+', '', text)
        text = re.sub(r'\S+@\S+', '', text)
        text = re.sub(r'[^a-z\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text).strip()
        logger.info(f"After cleaning: {text}")
        
        # Step 3: Enhanced Tokenization
        raw_tokens = re.findall(r'\b[a-z]+\b', text)
        
        tokens = []
        for token in raw_tokens:
            token = token.lower()
            
            # Keep single letters 'a' and 'i'
            if len(token) == 1 and token in ['a', 'i']:
                tokens.append(token)
            elif len(token) >= 2:
                # Remove repeated chars
                token = re.sub(r'(.)\1{2,}', r'\1\1', token)
                # Filter noise words
                if token not in {"thing", "something", "anything", "everything"}:
                    tokens.append(token)
        
        # Filter very long tokens (likely noise)
        tokens = [t for t in tokens if len(t) <= 50]
        
        logger.info(f"Tokens: {tokens}")
        
        # Step 4: Stopword removal (keep 'a' and 'i')
        if config.get("remove_stopwords", True):
            filtered_tokens = []
            for t in tokens:
                if t in ['a', 'i']:
                    filtered_tokens.append(t)
                elif t not in stop_words:
                    filtered_tokens.append(t)
            tokens = filtered_tokens
            logger.info(f"After stopword removal: {tokens}")
        
        # Step 5: Lemmatization
        if config.get("lemmatize", True) and tokens:
            doc = nlp(" ".join(tokens), disable=["parser", "ner"])
            tokens = [
                token.lemma_
                for token in doc
                if token.lemma_ != "-PRON-" and token.lemma_.strip()
            ]
            logger.info(f"After lemmatization: {tokens}")
        
        # Step 6: Remove duplicates
        if config.get("remove_duplicates", True):
            tokens = list(dict.fromkeys(tokens))
            logger.info(f"After duplicate removal: {tokens}")
        
        logger.info(f"Final Output: {tokens}")
        return tokens
        
    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}")
        return []

In [30]:
# comparing stemming vs lemmatization using your pipeline


def compare_stem_vs_lemma(text):
    if not validate_input(text):
        return None

    # Apply edge cases first
    text_processed = preprocess_edge_cases(text)
    
    print("\nRAW:", text)

    # Lemmatization (uses full enhanced pipeline)
    lemma = text_preprocessing_pipeline(text)

    # Stemming (also use edge cases)
    text_clean = clean_text(expand_contractions(text_processed))
    tokens = custom_tokenizer(text_clean)
    tokens = remove_stopwords(tokens)
    stemmed = [stemmer.stem(t) for t in tokens]

    print("\nSTEMMED:", stemmed)
    print("\nLEMMATIZED:", lemma)

    return {"stemmed": stemmed, "lemmatized": lemma}

In [31]:
# batch processing of multiple texts using the pipeline and also to get data from csv files.

def preprocess_batch(data, text_column=None, method="lemma"):

    # -----------------------
    # Load / extract texts
    # -----------------------
    if isinstance(data, str):
        data = pd.read_csv(data)

    if isinstance(data, pd.DataFrame):
        if text_column is None:
            raise ValueError("text_column must be provided for DataFrame input")
        texts = data[text_column].fillna("").tolist()

    elif isinstance(data, list):
        texts = data

    else:
        raise ValueError("Unsupported input type")

    # -----------------------
    # Processing
    # -----------------------
    results = []

    for text in texts:

        if not validate_input(text):
            results.append("")
            continue

        # 🔥 STEM MODE
        if method == "stem":
            text_clean = clean_text(expand_contractions(text))
            tokens = custom_tokenizer(text_clean)
            tokens = remove_stopwords(tokens)
            processed = [stemmer.stem(t) for t in tokens]

        # 🔥 LEMMA MODE (DEFAULT)
        else:
            processed = text_preprocessing_pipeline(text)

        results.append(" ".join(processed))

    return results

In [33]:
# Function to compare raw vs processed text side by side for better understanding of the transformations applied by the pipeline

def compare_raw_vs_processed(text):

    if not validate_input(text):
        print("Invalid input")
        return []

    print("\nRAW:", text)

    processed = text_preprocessing_pipeline(text)

    print("\nPROCESSED:", processed)

    return processed

In [34]:
# Function to run a simple command-line interface (CLI) for testing the pipeline with different options

def run_cli():
    while True:
        print("\n===== NLP PIPELINE =====")
        print("1. Single Text")
        print("2. Raw vs Processed")
        print("3. Stem vs Lemma")
        print("4. Batch CSV")
        print("5. Exit")

        choice = input("Enter choice: ")

        if choice == "1":
            text = input("Enter text: ")
            print(text_preprocessing_pipeline(text))

        elif choice == "2":
            text = input("Enter text: ")
            compare_raw_vs_processed(text)

        elif choice == "3":
            text = input("Enter text: ")
            compare_stem_vs_lemma(text)

        elif choice == "4":
            path = input("CSV path: ")
            col = input("Column name: ")

            df = pd.read_csv(path)
            df["processed_text"] = preprocess_batch(df, col)

            df.to_csv("processed_output.csv", index=False)
            print("Saved as processed_output.csv")

        elif choice == "5":
            break

        else:
            print("Invalid choice")

In [35]:
#test the edge cases to demonstrate the robustness of the pipeline, 
# we will create a test suite that covers a wide range of edge cases that can occur in real-world text data,
#  and we will run these tests to see how well our pipeline handles them.

def test_edge_cases():
    """Test suite for edge cases to demonstrate pipeline robustness"""
    
    test_cases = [
        ("Empty String", ""),
        ("None Value", None),
        ("Only Spaces", "   "),
        ("Single Letter 'a'", "a"),
        ("Single Letter 'I'", "I"),
        ("Repeated Characters", "coooool hiiiiiii"),
        ("Numbers in Words", "ai2024 version2 beta3"),
        ("Emojis", "I love 🎉 this! 😊"),
        ("URLs", "Check https://example.com for info"),
        ("Emails", "Contact me@test.com please"),
        ("HTML Tags", "<p>Hello</p> <b>World</b>"),
        ("Special Quotes", "Hello ‘world’ and “universe”"),
        ("Dashes", "This—is—awesome – really"),
        ("Ellipsis", "Wait... what... really?"),
        ("Smart Quotes", "It’s a beautiful day"),
        ("Very Long Word", "supercalifragilisticexpialidocious"),
        ("Only Punctuation", "!!! ??? ..."),
        ("Complex Contractions", "y'all'd've ain't gonna"),
        ("Social Media", "Check @username and #hashtag"),
        ("Mixed Case", "HELLO World Mixed CASE"),
        ("Mix of Everything", "COOOOOL!!! https://test.com @user #hash 🎉 It’s 2024!"),
    ]
    
    print("\n" + "="*80)
    print("EDGE CASE TEST SUITE RESULTS")
    print("="*80)
    
    results = []
    for name, test_input in test_cases:
        output = text_preprocessing_pipeline(test_input)
        status = "✅ PASS" if output is not None else "❌ FAIL"
        results.append({"test_case": name, "status": status, "output": output})
        
        print(f"\n📌 {name}:")
        print(f"   Input: {repr(test_input)[:60]}")
        print(f"   Output: {output}")
        print(f"   Status: {status}")
    
    print("\n" + "="*80)
    print(f"SUMMARY: {sum(1 for r in results if 'PASS' in r['status'])}/{len(results)} tests passed")
    print("="*80)
    
    return results

# Run the edge case tests
print("\n🔍 Running Edge Case Tests...\n")
test_results = test_edge_cases()


🔍 Running Edge Case Tests...


EDGE CASE TEST SUITE RESULTS

📌 Empty String:
   Input: ''
   Output: []
   Status: ✅ PASS

📌 None Value:
   Input: None
   Output: []
   Status: ✅ PASS

📌 Only Spaces:
   Input: '   '
   Output: []
   Status: ✅ PASS

📌 Single Letter 'a':
   Input: 'a'
   Output: ['a']
   Status: ✅ PASS

📌 Single Letter 'I':
   Input: 'I'
   Output: ['I']
   Status: ✅ PASS

📌 Repeated Characters:
   Input: 'coooool hiiiiiii'
   Output: ['cool', 'hii']
   Status: ✅ PASS

📌 Numbers in Words:
   Input: 'ai2024 version2 beta3'
   Output: ['ai', 'version', 'beta']
   Status: ✅ PASS

📌 Emojis:
   Input: 'I love 🎉 this! 😊'
   Output: ['I', 'love']
   Status: ✅ PASS

📌 URLs:
   Input: 'Check https://example.com for info'
   Output: ['check', 'info']
   Status: ✅ PASS

📌 Emails:
   Input: 'Contact me@test.com please'
   Output: ['contact', 'please']
   Status: ✅ PASS

📌 HTML Tags:
   Input: '<p>Hello</p> <b>World</b>'
   Output: ['hello', 'world']
   Status: ✅ PASS

📌 Special Quo

In [36]:
# Quick verification cell - add this after all changes
print("Testing edge cases quickly...")

test_texts = [
    ("coooool!!!", "repeated chars"),
    ("ai2024 version2", "numbers in words"),
    ("It’s a 🎉 day!", "emojis + smart quotes"),
]

for text, desc in test_texts:
    result = text_preprocessing_pipeline(text)
    print(f"{desc:20} | Input: {text:30} | Output: {result}")

Testing edge cases quickly...
repeated chars       | Input: coooool!!!                     | Output: ['cool']
numbers in words     | Input: ai2024 version2                | Output: ['ai', 'version']
emojis + smart quotes | Input: It’s a 🎉 day!                  | Output: ['a', 'day']


In [37]:
# Interactive demo to test edge cases in real-time, 
# this allows us to see how the pipeline handles various inputs and gives us a chance to experiment with different edge cases on the fly.

def demo_edge_cases():
    """Interactive demo to test custom edge cases"""
    print("\n" + "="*60)
    print("🔧 EDGE CASE TESTING DEMO")
    print("="*60)
    print("\nEnter any text to see how the pipeline handles it.")
    print("Try these examples:")
    print("  - 'coooool!!! https://test.com 🎉'")
    print("  - 'ai2024 version2.0'")
    print("  - " + '"It’s a beautiful day—really! 😊"')
    print("  - 'y'all'd've'")
    print("\nType 'quit' to exit\n")
    
    while True:
        user_input = input("👉 Enter text to test: ")
        if user_input.lower() == 'quit':
            print("Exiting demo...")
            break
        
        if user_input.strip():
            print(f"\n📝 INPUT: {user_input}")
            result = text_preprocessing_pipeline(user_input)
            print(f"✨ OUTPUT: {result}\n")
        else:
            print("⚠️ Please enter some text or 'quit' to exit\n")

# Uncomment to run interactive demo:
# demo_edge_cases()